# 03 · Explainability — Grad-CAM-1D and Integrated Gradients

**Goal**: visualise what the trained classifier looks at when calling AFib.

**Prereq**: a checkpoint at `outputs/checkpoints/best.ckpt` (run `af-train --epochs 50` first).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from af_explain.data.dataset import LABEL_NAMES, PhysioNet2017Dataset
from af_explain.explain.gradcam import make_gradcam
from af_explain.explain.integrated_gradients import explain_with_ig
from af_explain.training.lightning_module import AFClassifier

DATA_ROOT = Path('../data/raw/training2017')
CKPT = Path('../outputs/checkpoints/best.ckpt')
assert CKPT.exists(), f'No checkpoint at {CKPT}. Train first.'

In [ ]:
model = AFClassifier.load_from_checkpoint(str(CKPT), map_location='cpu')
model.eval()
test_ds = PhysioNet2017Dataset(DATA_ROOT, split='test', sample_mode='center')

## 1. Pick one correctly-classified record per class

In [ ]:
samples_per_class: dict[int, dict] = {}
with torch.no_grad():
    for idx in range(len(test_ds)):
        item = test_ds[idx]
        y = int(item['label'])
        if y in samples_per_class:
            continue
        x = item['signal'].unsqueeze(0)
        pred = int(model(x).argmax(1).item())
        if pred == y:
            samples_per_class[y] = {'idx': idx, 'signal': item['signal'], 'label': y}
        if len(samples_per_class) == 4:
            break
{LABEL_NAMES[k]: v['idx'] for k, v in samples_per_class.items()}

## 2. Grad-CAM and IG side by side

In [ ]:
FS = 300
fig, axes = plt.subplots(4, 3, figsize=(16, 12), sharex=True)

for row_idx, (cls, sample) in enumerate(samples_per_class.items()):
    sig = sample['signal']  # (1, T)
    tensor = sig.unsqueeze(0)
    t = np.arange(sig.shape[-1]) / FS

    cam = make_gradcam(model.model)
    saliency = cam(tensor, target_class=cls)
    cam.remove_hooks()
    ig = explain_with_ig(model.model, tensor, target_class=cls)

    axes[row_idx, 0].plot(t, sig[0].numpy(), lw=0.6, color='black')
    axes[row_idx, 0].set_title(f'{LABEL_NAMES[cls]} · ECG')
    axes[row_idx, 1].plot(t, saliency, color='crimson')
    axes[row_idx, 1].set_title('Grad-CAM (window-level)')
    axes[row_idx, 2].plot(t, ig, color='steelblue')
    axes[row_idx, 2].axhline(0, color='gray', ls=':')
    axes[row_idx, 2].set_title('Integrated Gradients (sample-level, signed)')
for ax in axes[-1]:
    ax.set_xlabel('seconds')
plt.tight_layout()

## TODO — clinical reading (your turn)

For each row:
1. Is the highlighted region morphologically what you expect for that rhythm?
   - AFib → irregular RR, absent P, fibrillatory baseline.
   - Normal → regular sinus, visible P preceding QRS.
2. Where the model and your gaze disagree, note the record_id in `annotations/` — these become evidence for the concordance analysis.